# 第7讲：主成分分析与机器学习简介（学生操练）

        > 课程：《交通大数据分析与应用》  
        > 数据：课程模拟数据，不是实际监测数据  
        > 建议用时：11分钟

        ## 目标

        1. 把N_COMPONENTS设为2和3，比较累计贡献率与压缩程度；
2. 找出PC1、PC2绝对载荷最大的变量，并给出有依据的名称；
3. 把THRESHOLD设为0.35和0.65，比较Precision、Recall与报警数量；

        学生版的代码可以直接运行；请按`TODO`修改参数、核对输出并完成解释。教师版和参考答案版已保存一次完整运行结果。

## 1. Setup｜环境、路径与参数

In [ ]:
from __future__ import annotations

from pathlib import Path
import warnings

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)
DATA_DIR = Path("data")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
plt.rcParams.update({"figure.dpi": 120, "axes.grid": True, "grid.alpha": 0.25})
RANDOM_STATE = 42
print("环境已就绪；数据目录：", DATA_DIR.resolve())

## 2. 建立检测器—日运行画像

In [ ]:
traffic = pd.read_csv(DATA_DIR / "traffic_15min.csv", parse_dates=["timestamp"]).sort_values(["timestamp", "detector_id"])
profile = (traffic.assign(date=traffic["timestamp"].dt.date)
           .groupby(["detector_id", "date"], as_index=False)
           .agg(mean_flow=("flow_15min", "mean"),
                mean_speed=("speed_kmh", "mean"),
                min_speed=("speed_kmh", "min"),
                speed_sd=("speed_kmh", "std"),
                mean_occupancy=("occupancy_pct", "mean"),
                congestion_rate=("congestion", "mean"),
                severe_rate=("severe_congestion", "mean")))
profile_features = ["mean_flow", "mean_speed", "min_speed", "speed_sd", "mean_occupancy", "congestion_rate", "severe_rate"]
print("画像矩阵", profile.shape, "样本×变量", len(profile), "×", len(profile_features))
display(profile.head())

## 3. 标准化并进行主成分分析

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

N_COMPONENTS = 2  # TODO：改为3，比较信息保留与压缩程度
profile_scaler = StandardScaler()
profile_scaled = profile_scaler.fit_transform(profile[profile_features])
full_pca = PCA(random_state=RANDOM_STATE).fit(profile_scaled)
all_explained = pd.DataFrame({
    "component": [f"PC{i+1}" for i in range(len(profile_features))],
    "ratio": full_pca.explained_variance_ratio_,
    "cumulative": full_pca.explained_variance_ratio_.cumsum(),
})
pca = PCA(n_components=N_COMPONENTS, random_state=RANDOM_STATE).fit(profile_scaled)
scores = pca.transform(profile_scaled)
loadings = pd.DataFrame(pca.components_.T, index=profile_features,
                        columns=[f"PC{i+1}" for i in range(N_COMPONENTS)])
print("保留维数", N_COMPONENTS, "累计贡献率", round(pca.explained_variance_ratio_.sum(), 4))
display(all_explained.round(4)); display(loadings.round(3))
all_explained.to_csv(OUTPUT_DIR / "lesson07_explained_variance.csv", index=False)
loadings.to_csv(OUTPUT_DIR / "lesson07_loadings.csv")

## 4. 由载荷解释主成分

In [ ]:
dominant = pd.DataFrame({
    pc: [f"{name} ({loadings.loc[name, pc]:+.3f})" for name in loadings[pc].abs().nlargest(3).index]
    for pc in loadings.columns
})
display(dominant)
ax = all_explained.plot(x="component", y=["ratio", "cumulative"], marker="o",
                        ylim=(0, 1.05), figsize=(8.5, 4.5),
                        title="PCA explained variance")
ax.axhline(.90, color="#C30D23", linestyle="--", linewidth=1)
ax.set_ylabel("ratio"); plt.tight_layout()
plt.savefig(OUTPUT_DIR / "lesson07_pca_variance.png"); plt.show()

## 5. 按时间训练分类Pipeline并比较阈值

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (confusion_matrix, precision_score, recall_score,
                             f1_score, roc_auc_score, average_precision_score,
                             precision_recall_curve, roc_curve)

data = traffic.copy()
data["next_severe"] = data.groupby("detector_id")["severe_congestion"].shift(-1)
data = data.dropna(subset=["next_severe"]).copy()
data["next_severe"] = data["next_severe"].astype(int)
raw_features = ["flow_15min", "occupancy_pct", "rain", "is_peak", "lanes"]
train = data[data["timestamp"] < "2026-03-10"]
valid = data[(data["timestamp"] >= "2026-03-10") & (data["timestamp"] < "2026-03-12")]
test = data[data["timestamp"] >= "2026-03-12"]

pipe = Pipeline([
    ("scale", StandardScaler()),
    ("pca", PCA(n_components=min(N_COMPONENTS, len(raw_features)), random_state=RANDOM_STATE)),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE)),
])
pipe.fit(train[raw_features], train["next_severe"])
valid_risk = pipe.predict_proba(valid[raw_features])[:, 1]
rows = []
for threshold in [0.35, 0.50, 0.65]:
    pred = (valid_risk >= threshold).astype(int)
    rows.append({
        "threshold": threshold,
        "precision": precision_score(valid["next_severe"], pred, zero_division=0),
        "recall": recall_score(valid["next_severe"], pred, zero_division=0),
        "f1": f1_score(valid["next_severe"], pred, zero_division=0),
        "alerts": int(pred.sum()),
    })
threshold_table = pd.DataFrame(rows)
display(threshold_table.round(3))
threshold_table.to_csv(OUTPUT_DIR / "lesson07_threshold_comparison.csv", index=False)

valid_precision, valid_recall, _ = precision_recall_curve(valid["next_severe"], valid_risk)
valid_fpr, valid_tpr, _ = roc_curve(valid["next_severe"], valid_risk)
valid_ap = average_precision_score(valid["next_severe"], valid_risk)
valid_auc = roc_auc_score(valid["next_severe"], valid_risk)
fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
axes[0].plot(valid_recall, valid_precision, color="#3E6A8D", linewidth=2,
             label=f"AP={valid_ap:.3f}")
axes[0].axhline(valid["next_severe"].mean(), color="#8B9096", linestyle="--",
                label="random baseline")
axes[0].set(xlabel="Recall", ylabel="Precision", title="Validation P-R curve",
            xlim=(0, 1.02), ylim=(0, 1.02)); axes[0].legend()
axes[1].plot(valid_fpr, valid_tpr, color="#3E7C78", linewidth=2,
             label=f"AUC={valid_auc:.3f}")
axes[1].plot([0, 1], [0, 1], color="#8B9096", linestyle="--",
             label="random baseline")
axes[1].set(xlabel="False positive rate", ylabel="True positive rate",
            title="Validation ROC curve", xlim=(0, 1.02), ylim=(0, 1.02))
axes[1].legend(); plt.tight_layout()
plt.savefig(OUTPUT_DIR / "lesson07_pr_roc.png"); plt.show()

THRESHOLD = 0.50  # TODO：根据验证期结果改为0.35或0.65
test_risk = pipe.predict_proba(test[raw_features])[:, 1]
test_pred = (test_risk >= THRESHOLD).astype(int)
test_cm = pd.DataFrame(confusion_matrix(test["next_severe"], test_pred),
                       index=["actual_0", "actual_1"], columns=["pred_0", "pred_1"])
test_auc = roc_auc_score(test["next_severe"], test_risk)
print("测试期AUC", round(test_auc, 3), "最终阈值", THRESHOLD); display(test_cm)
test_cm.to_csv(OUTPUT_DIR / "lesson07_test_confusion.csv")

checks = {
    "画像样本为84": len(profile) == 84,
    "前两个主成分累计超过0.90": all_explained.loc[1, "cumulative"] > .90,
    "训练早于验证和测试": train["timestamp"].max() < valid["timestamp"].min() < test["timestamp"].min(),
    "Pipeline包含标准化和PCA": list(pipe.named_steps)[:2] == ["scale", "pca"],
    "六个输出文件已生成": all((OUTPUT_DIR / name).exists() for name in [
        "lesson07_explained_variance.csv", "lesson07_loadings.csv",
        "lesson07_pca_variance.png", "lesson07_threshold_comparison.csv",
        "lesson07_pr_roc.png", "lesson07_test_confusion.csv"]),
}
status = "PASS" if all(checks.values()) else "CHECK"
(OUTPUT_DIR / "自检结果.txt").write_text(status, encoding="utf-8")
print(status, checks)

## Checks｜当堂记录

        - 把N_COMPONENTS设为2和3，比较累计贡献率与压缩程度
- 找出PC1、PC2绝对载荷最大的变量，并给出有依据的名称
- 把THRESHOLD设为0.35和0.65，比较Precision、Recall与报警数量

        **预期结果：** 贡献率表与图、载荷表、验证期阈值比较表和P-R／ROC曲线、测试期混淆矩阵。

        **完成标准：** 主成分命名来自载荷；阈值选择使用验证期；测试期只做最终评价；Notebook生成PASS。

        请在课堂记录中写下：改了什么参数、结果发生了什么变化、这个变化在交通问题中意味着什么。